# 04b — Domain Adaptation Extensions (80/20 Design)

**Parallel to `04_model_extensions.ipynb` — same extensions, different data and base model.**

**Inputs:**
- `_80` preprocessed files from `02b_data_preprocessing_8020.ipynb`
- `rf_calibrated_80.pkl` from `03b_baseline_model_8020.ipynb`

**Extensions:**
| # | Name | Description |
|---|---|---|
| A | Label Shift Correction | Post-hoc log-odds prior adjustment |
| B | IW + Label Shift | XGBoost domain classifier → importance weights → RF retrain → LS wrapper |

**Output:** `models/extension_models_80.pkl`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q imbalanced-learn xgboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os, pickle, json
warnings.filterwarnings('ignore')

from sklearn.ensemble        import RandomForestClassifier
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics         import (roc_auc_score, average_precision_score,
                                     roc_curve, confusion_matrix)
from xgboost                 import XGBClassifier

SEED = 42
np.random.seed(SEED)

BASE      = '/content/drive/MyDrive/AI in Medicine/data/output_data/preprocessed'
MODEL_DIR = '/content/drive/MyDrive/AI in Medicine/models'
EXT_DIR   = f'{MODEL_DIR}/extensions_80'
os.makedirs(EXT_DIR, exist_ok=True)

print('Libraries loaded.')

Libraries loaded.


## 1. Load 80/20 Data and Base Model

In [ ]:
X_train = pd.read_csv(f'{BASE}/X_train_80.csv')
y_train = pd.read_csv(f'{BASE}/y_train_80.csv').squeeze()
X_val   = pd.read_csv(f'{BASE}/X_val_80.csv')
y_val   = pd.read_csv(f'{BASE}/y_val_80.csv').squeeze()
X_mimic = pd.read_csv(f'{BASE}/X_mimic_80.csv')
y_mimic = pd.read_csv(f'{BASE}/y_mimic_80.csv').squeeze()

with open(f'{MODEL_DIR}/rf_calibrated_80.pkl', 'rb') as f:
    rf_calibrated = pickle.load(f)
with open(f'{MODEL_DIR}/model_params_80.json') as f:
    params = json.load(f)

rf_best_params   = params['rf_best_params']
rf_cal_thr       = params['rf_cal_threshold']
scale_pos        = params['scale_pos']
prior_log_odds   = params['log_odds_shift']
mimic_threshold  = params['mimic_threshold']

print(f'X_train : {X_train.shape}   mortality={y_train.mean():.3f}')
print(f'X_val   : {X_val.shape}     mortality={y_val.mean():.3f}')
print(f'X_mimic : {X_mimic.shape}   mortality={y_mimic.mean():.3f}')
print(f'rf_cal_threshold: {rf_cal_thr:.4f}')
print(f'log_odds_shift  : {prior_log_odds:.4f}')

X_train : (2016, 81)   mortality=0.050
X_val   : (504, 81)     mortality=0.050
X_mimic : (136, 81)   mortality=0.338
rf_cal_threshold: 0.0993
log_odds_shift  : 2.2712


## 2. Evaluation Utilities

In [ ]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

class LabelShiftModel:
    def __init__(self, base_model, adjustment):
        self.base_model  = base_model
        self.adjustment  = adjustment
    def predict_proba(self, X):
        raw = self.base_model.predict_proba(X)[:, 1].clip(1e-6, 1 - 1e-6)
        lo  = np.log(raw / (1 - raw)) + self.adjustment
        p   = 1.0 / (1.0 + np.exp(-lo))
        return np.vstack([1 - p, p]).T

def evaluate_model(model, X, y_true, label='', threshold=0.5):
    proba = model.predict_proba(X)[:, 1]
    pred  = (proba >= threshold).astype(int)
    cm    = confusion_matrix(y_true, pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, int(y_true.sum())
    return {
        'label'      : label,
        'ROC-AUC'    : roc_auc_score(y_true, proba),
        'PR-AUC'     : average_precision_score(y_true, proba),
        'Sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'threshold'  : threshold,
        'proba'      : proba,
        'y_true'     : np.array(y_true),
    }

def threshold_min_sensitivity(model, X_v, y_v, min_sens=0.85):
    p = model.predict_proba(X_v)[:, 1]
    fpr, tpr, thr = roc_curve(y_v, p)
    mask = tpr >= min_sens
    if mask.any():
        return float(thr[mask][-1])
    return float(thr[np.argmax(tpr - fpr)])

def print_row(label, r_val, r_mimic):
    print(f'{label:<35}  '
          f'val={r_val["ROC-AUC"]:.3f}/{r_val["PR-AUC"]:.3f}  '
          f'mimic={r_mimic["ROC-AUC"]:.3f}/{r_mimic["PR-AUC"]:.3f}  '
          f'sens={r_mimic["Sensitivity"]:.3f}  spec={r_mimic["Specificity"]:.3f}')

print('Utilities ready.')

Utilities ready.


## 3. Baseline Reference

In [ ]:
base_val   = evaluate_model(rf_calibrated, X_val,   y_val,   label='Baseline',       threshold=rf_cal_thr)
base_mimic = evaluate_model(rf_calibrated, X_mimic, y_mimic, label='A0 Baseline RF', threshold=mimic_threshold)

print('Baseline — 80/20 model:')
print(f'  Val   AUC={base_val["ROC-AUC"]:.3f}  Sens={base_val["Sensitivity"]:.3f}  Spec={base_val["Specificity"]:.3f}')
print(f'  MIMIC AUC={base_mimic["ROC-AUC"]:.3f}  Sens={base_mimic["Sensitivity"]:.3f}  Spec={base_mimic["Specificity"]:.3f}')

Baseline — 80/20 model:
  Val   AUC=0.826  Sens=0.640  Spec=0.904
  MIMIC AUC=0.734  Sens=0.217  Spec=0.989


## Extension A — Label Shift Correction

In [ ]:
ls_model_mimic = LabelShiftModel(rf_calibrated, prior_log_odds)
ls_model_zero  = LabelShiftModel(rf_calibrated, 0.0)

ls_thr_val = threshold_min_sensitivity(ls_model_zero, X_val, y_val, min_sens=0.85)

ls_val   = evaluate_model(ls_model_zero,  X_val,   y_val,   label='A LS',        threshold=ls_thr_val)
ls_mimic = evaluate_model(ls_model_mimic, X_mimic, y_mimic, label='A LS (MIMIC)', threshold=mimic_threshold)

print_row('A  Label Shift only', ls_val, ls_mimic)

A  Label Shift only                  val=0.826/0.432  mimic=0.734/0.642  sens=0.674  spec=0.678


## Extension B — Importance Weighting + Label Shift

In [ ]:
print('Step B1: XGBoost domain classifier (eICU=0 vs MIMIC=1)')
X_combined = np.vstack([X_train.values, X_mimic.values])
d_labels   = np.array([0] * len(X_train) + [1] * len(X_mimic))

domain_xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', random_state=SEED, n_jobs=-1
)
domain_xgb.fit(X_combined, d_labels)
domain_auc = roc_auc_score(d_labels, domain_xgb.predict_proba(X_combined)[:, 1])
print(f'Domain classifier AUC: {domain_auc:.3f}  (>0.7 = significant covariate shift)')

Step B1: XGBoost domain classifier (eICU=0 vs MIMIC=1)
Domain classifier AUC: 1.000  (>0.7 = significant covariate shift)


In [ ]:
print('Step B2: Importance weights')
p_target   = domain_xgb.predict_proba(X_train.values)[:, 1]
iw_weights = np.clip(p_target / (1 - p_target + 1e-6), 0.05, 20.0)
iw_weights = iw_weights / iw_weights.mean()
print(f'IW weights — min:{iw_weights.min():.3f}  max:{iw_weights.max():.3f}  '
      f'mean:{iw_weights.mean():.3f}  std:{iw_weights.std():.3f}')

print('\nStep B3: Retrain RF with importance weights')
rf_iw_base = RandomForestClassifier(**rf_best_params, class_weight='balanced',
                                    random_state=SEED, n_jobs=-1)
rf_iw_cal  = CalibratedClassifierCV(rf_iw_base, method='sigmoid', cv=cv_strategy)
rf_iw_cal.fit(X_train, y_train, sample_weight=iw_weights.astype(np.float64))
print('RF + IW trained and calibrated.')

print('\nStep B4: Wrap with Label Shift correction')
iw_ls_model = LabelShiftModel(rf_iw_cal, prior_log_odds)
iw_ls_zero  = LabelShiftModel(rf_iw_cal, 0.0)
iw_ls_thr   = threshold_min_sensitivity(iw_ls_zero, X_val, y_val, min_sens=0.85)

iw_ls_val   = evaluate_model(iw_ls_zero,  X_val,   y_val,   label='B IW+LS',        threshold=iw_ls_thr)
iw_ls_mimic = evaluate_model(iw_ls_model, X_mimic, y_mimic, label='B IW+LS (MIMIC)', threshold=mimic_threshold)

print_row('B  IW + Label Shift', iw_ls_val, iw_ls_mimic)

Step B2: Importance weights
IW weights — min:0.997  max:2.587  mean:1.000  std:0.057

Step B3: Retrain RF with importance weights
RF + IW trained and calibrated.

Step B4: Wrap with Label Shift correction
B  IW + Label Shift                  val=0.829/0.428  mimic=0.734/0.642  sens=0.696  spec=0.656


## 4. Comparison Table

In [ ]:
all_methods = [
    ('A0 Baseline RF',      base_val,   base_mimic),
    ('A  Label Shift',      ls_val,     ls_mimic),
    ('B  IW + Label Shift', iw_ls_val,  iw_ls_mimic),
]

print('=' * 95)
print(f'{"Method":<28}  {"Val AUC":>8}  {"MIMIC AUC":>10}  '
      f'{"MIMIC PR":>9}  {"MIMIC Sens":>11}  {"MIMIC Spec":>11}')
print('-' * 95)
for label, r_v, r_m in all_methods:
    print(f'{label:<28}  '
          f'{r_v["ROC-AUC"]:>8.3f}  '
          f'{r_m["ROC-AUC"]:>10.3f}  '
          f'{r_m["PR-AUC"]:>9.3f}  '
          f'{r_m["Sensitivity"]:>11.3f}  '
          f'{r_m["Specificity"]:>11.3f}')
print('=' * 95)

print('\n── Δ over Baseline on MIMIC ──')
base_auc  = base_mimic['ROC-AUC']
base_sens = base_mimic['Sensitivity']
for label, _, r_m in all_methods[1:]:
    d_auc  = r_m['ROC-AUC']    - base_auc
    d_sens = r_m['Sensitivity'] - base_sens
    print(f'  {label:<28}  ΔAUC={d_auc:+.3f}   ΔSens={d_sens:+.3f}')

Method                         Val AUC   MIMIC AUC   MIMIC PR   MIMIC Sens   MIMIC Spec
-----------------------------------------------------------------------------------------------
A0 Baseline RF                   0.826       0.734      0.642        0.217        0.989
A  Label Shift                   0.826       0.734      0.642        0.674        0.678
B  IW + Label Shift              0.829       0.734      0.642        0.696        0.656

── Δ over Baseline on MIMIC ──
  A  Label Shift                ΔAUC=+0.000   ΔSens=+0.457
  B  IW + Label Shift           ΔAUC=+0.000   ΔSens=+0.478


## 5. Save Extension Models

In [ ]:
extension_80 = {
    # Extension B
    'iw_model'        : rf_iw_cal,
    'iw_weights'      : iw_weights,
    'domain_xgb'      : domain_xgb,
    'iw_ls_threshold' : iw_ls_thr,
    'ls_threshold'    : ls_thr_val,
    # Shared
    'prior_log_odds'  : prior_log_odds,
    'mimic_threshold' : mimic_threshold,
    # Summary
    'results_mimic'   : {
        label: {'ROC-AUC': r_m['ROC-AUC'], 'Sensitivity': r_m['Sensitivity']}
        for label, _, r_m in all_methods
    },
}

save_path = f'{EXT_DIR}/extension_models_80.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(extension_80, f)

print(f'Saved → {save_path}')
for k in extension_80.keys():
    print(f'  {k}')

Saved → /content/drive/MyDrive/AI in Medicine/models/extensions_80/extension_models_80.pkl
  iw_model
  iw_weights
  domain_xgb
  iw_ls_threshold
  ls_threshold
  prior_log_odds
  mimic_threshold
  results_mimic
